In [1]:
!nvidia-smi
import torch
print("Torch:", torch.__version__)
print("CUDA dispo:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM totale (Go):", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))

Sat Nov 14 01:01:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub sentencepiece

In [4]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))
print("Connecté à Hugging Face")

Connecté à Hugging Face


In [5]:
from huggingface_hub import snapshot_download

MODELS = {
    "qwen": "Qwen/Qwen2.5-3B-Instruct",
    "phi":  "microsoft/Phi-3.5-mini-instruct",
}
for name, mid in MODELS.items():
    path = snapshot_download(mid)
    print(name, "->", path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

qwen -> /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

phi -> /root/.cache/huggingface/hub/models--microsoft--Phi-3.5-mini-instruct/snapshots/2fe192450127e6a83f7441aef6e3ca586c338b77


In [6]:
import time, gc, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SYSTEM = "You are a cybersecurity SOC assistant. Answer concisely."
PROMPT = ("Alert: 15 failed SSH logins from 185.220.101.4 on host srv-db01 within 2 minutes, "
          "followed by one successful login as root. Classify the severity, "
          "map it to a MITRE ATT&CK technique, and recommend an action.")

def test_model(name, model_id, max_new_tokens=200):
    tok = AutoTokenizer.from_pretrained(model_id)
    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="auto")
    load_time = time.time() - t0
    mem_model = torch.cuda.memory_allocated() / 1024**3

    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": PROMPT}]
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(model.device)

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    latency = time.time() - t0

    n_in = inputs["input_ids"].shape[1]
    n_new = out.shape[1] - n_in
    answer = tok.decode(out[0][n_in:], skip_special_tokens=True)

    res = {
        "model": model_id,
        "load_time_s": round(load_time, 1),
        "vram_model_gb": round(mem_model, 2),
        "vram_peak_gb": round(torch.cuda.max_memory_allocated()/1024**3, 2),
        "latency_s": round(latency, 2),
        "new_tokens": n_new,
        "tokens_per_s": round(n_new / latency, 1),
        "answer": answer,
    }
    del model, tok, inputs, out
    gc.collect(); torch.cuda.empty_cache()
    return res

results = {}
for name, mid in MODELS.items():
    results[name] = test_model(name, mid)
    print("=" * 60); print(name.upper())
    for k, v in results[name].items():
        print(f"{k}: {v}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


QWEN
model: Qwen/Qwen2.5-3B-Instruct
load_time_s: 43.5
vram_model_gb: 5.75
vram_peak_gb: 5.77
latency_s: 7.95
new_tokens: 143
tokens_per_s: 18.0
answer: Severity: High

MITRE ATT&CK Technique: T1071 - Command and Scripting Interpreter (e.g., using Bash)

Action: 
1. Implement rate limiting or IP blocking for SSH login attempts from 185.220.101.4.
2. Review the server's SSH configuration to ensure it is set up securely (e.g., disable root login via SSH).
3. Conduct a forensic analysis of the successful login event to identify if it was an insider threat or a legitimate user.
4. Educate users about phishing and social engineering tactics that could lead to such incidents.
5. Consider implementing multi-factor authentication (MFA) to enhance security.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

PHI
model: microsoft/Phi-3.5-mini-instruct
load_time_s: 33.1
vram_model_gb: 7.13
vram_peak_gb: 7.24
latency_s: 8.73
new_tokens: 200
tokens_per_s: 22.9
answer: Severity: High

MITRE ATT&CK Technique: T1110 - Brute Force (Credentials)

Recommended Action:
1. Immediately block the IP address 185.220.101.4 at the firewall level to prevent further attempts.
2. Investigate the failed login attempts to determine if they are part of a larger attack pattern or if they are random attempts.
3. Review and strengthen SSH authentication mechanisms, such as implementing key-based authentication, disabling root login, and enforcing account lockout policies.
4. Monitor for any further suspicious activities and consider implementing an intrusion detection system (IDS) or intrusion prevention system (IPS) for better detection and response.
5. Conduct a security audit to ensure that all systems are up to date with the latest patches and that proper security measures are in place.



In [7]:
with open("week1_baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)